# Procesamiento de Lenguaje Natural: Frankenstein
## Parte 2: Vectorización y Representación Semántica

**Objetivos:**
- Implementar Bag-of-Words (BoW)
- Implementar TF-IDF
- Visualizar en espacio vectorial 3D
- Comparar representaciones


## 0. Importar Librerías

In [ ]:
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

print("✓ Todas las librerías importadas correctamente")

## 1. Cargar Modelos y Texto

In [ ]:
# Cargar modelo de spaCy
print("[1] Cargando modelo de spaCy...")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("✓ Modelo cargado")

# Cargar texto
print("\n[2] Cargando Frankenstein...")
with open("libro.txt", "r", encoding="utf-8") as f:
    texto_frankenstein = f.read()

print(f"✓ Texto cargado: {len(texto_frankenstein):,} caracteres")

## 2. Procesamiento: Tokenización y Lematización

In [ ]:
print("[3] Procesando texto con spaCy...")
doc = nlp(texto_frankenstein)

print(f"✓ Tokenización completada: {len(doc):,} tokens")
print(f"  Primeros 20 tokens: {[token.text for token in doc][:20]}")

## 3. Crear Corpus Lematizado (Por Oraciones)

In [ ]:
print("[4] Lematizando por oraciones...")

corpus_lematizado = []

# Iterar por oraciones (sentences)
for oracion in doc.sents:
    # Lematizar cada token
    lemas_oracion = [
        token.lemma_.lower()
        for token in oracion
        if not token.is_punct and not token.is_space and not token.is_stop
    ]
    
    # Agregar si tiene contenido
    if lemas_oracion:
        corpus_lematizado.append(" ".join(lemas_oracion))

print(f"✓ Corpus creado: {len(corpus_lematizado)} oraciones")
print(f"\nPrimeras 5 oraciones procesadas:")
for i, oracion in enumerate(corpus_lematizado[:5]):
    print(f"  {i+1}. {oracion[:80]}...")

## 4. Vectorización: Bag-of-Words (BoW)

In [ ]:
print("[5] Creando representación Bag-of-Words...\n")

# Crear vectorizador BoW
bow_vectorizer = CountVectorizer(max_features=100)  # Limitamos a 100 palabras más frecuentes
X_bow = bow_vectorizer.fit_transform(corpus_lematizado)
vocab_bow = bow_vectorizer.get_feature_names_out()

print(f"✓ Bag-of-Words completado")
print(f"  - Forma de matriz: {X_bow.shape}")
print(f"  - Oraciones: {X_bow.shape[0]}")
print(f"  - Dimensiones (palabras): {X_bow.shape[1]}")
print(f"\nPalabras más frecuentes (primeras 20):")
print(f"  {list(vocab_bow[:20])}")

# Mostrar matriz densa (sample)
print(f"\nMuestra de matriz BoW (primeras 5 oraciones, primeras 10 palabras):")
print(pd.DataFrame(X_bow[:5, :10].toarray(), columns=vocab_bow[:10]))

## 5. Vectorización: TF-IDF

In [ ]:
print("[6] Creando representación TF-IDF...\n")

# Crear vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=100)  # Mismo límite para comparación
X_tfidf = tfidf_vectorizer.fit_transform(corpus_lematizado)
vocab_tfidf = tfidf_vectorizer.get_feature_names_out()

print(f"✓ TF-IDF completado")
print(f"  - Forma de matriz: {X_tfidf.shape}")
print(f"  - Oraciones: {X_tfidf.shape[0]}")
print(f"  - Dimensiones (palabras): {X_tfidf.shape[1]}")
print(f"\nPalabras más importantes (primeras 20):")
print(f"  {list(vocab_tfidf[:20])}")

# Mostrar matriz densa (sample)
print(f"\nMuestra de matriz TF-IDF (primeras 5 oraciones, primeras 10 palabras):")
print(pd.DataFrame(X_tfidf[:5, :10].toarray(), columns=vocab_tfidf[:10]).round(4))

## 6. Comparación BoW vs TF-IDF

In [ ]:
print("[7] Análisis Comparativo\n")

# Estadísticas generales
print("Estadísticas Bag-of-Words:")
print(f"  - Total de conteos: {X_bow.sum():,.0f}")
print(f"  - Promedio por oración: {X_bow.sum() / X_bow.shape[0]:.2f}")
print(f"  - Densidad: {X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]) * 100:.2f}%")

print("\nEstadísticas TF-IDF:")
print(f"  - Suma total: {X_tfidf.sum():,.2f}")
print(f"  - Promedio por oración: {X_tfidf.sum() / X_tfidf.shape[0]:.4f}")
print(f"  - Densidad: {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]) * 100:.2f}%")

# Palabras top según cada método
print("\n" + "="*60)
print("Palabras TOP según cada método:")
print("="*60)

# Top BoW
bow_sums = np.array(X_bow.sum(axis=0)).ravel()
top_bow_idx = np.argsort(bow_sums)[-10:][::-1]
print("\nTop 10 Bag-of-Words (por frecuencia):")
for idx in top_bow_idx:
    print(f"  {vocab_bow[idx]:20} → {bow_sums[idx]:7.0f} apariciones")

# Top TF-IDF
tfidf_sums = np.array(X_tfidf.sum(axis=0)).ravel()
top_tfidf_idx = np.argsort(tfidf_sums)[-10:][::-1]
print("\nTop 10 TF-IDF (por importancia):")
for idx in top_tfidf_idx:
    print(f"  {vocab_tfidf[idx]:20} → {tfidf_sums[idx]:7.2f} importancia")

## 7. Visualización 3D: Función Auxiliar

In [ ]:
def graficar_palabras_3d(ax, matriz, vocabulario, titulo, color_puntos):
    """
    Grafica palabras en espacio vectorial 3D usando PCA
    
    Parámetros:
    - ax: eje 3D de matplotlib
    - matriz: matriz sparse de vectorización (oraciones x palabras)
    - vocabulario: nombres de las palabras
    - titulo: título del gráfico
    - color_puntos: color de los puntos
    """
    
    # 1. TRANSPONER: Filas = Palabras, Columnas = Contextos (oraciones)
    matriz_palabras = matriz.T
    
    # 2. PCA: Reducir a 3 DIMENSIONES
    pca = PCA(n_components=3)
    coords = pca.fit_transform(matriz_palabras.toarray())
    
    # Extraer coordenadas X, Y, Z
    x = coords[:, 0]
    y = coords[:, 1]
    z = coords[:, 2]
    
    # 3. GRAFICAR SCATTER 3D
    ax.scatter(x, y, z, c=color_puntos, s=100, edgecolors='black', 
              alpha=0.8, depthshade=True, linewidth=0.5)
    
    # 4. ETIQUETAR PUNTOS
    for i, palabra in enumerate(vocabulario):
        ax.text(x[i], y[i], z[i] + 0.05, palabra, fontsize=8, 
               ha='center', weight='bold')
    
    # 5. CONFIGURAR EJES
    ax.set_title(titulo, fontsize=12, weight='bold', pad=20)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=10)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=10)
    ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.1%})', fontsize=10)
    
    # Líneas de referencia (ejes coordenados)
    ax.plot([0,0], [0,0], [z.min(), z.max()], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([x.min(), x.max()], [0,0], [0,0], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([0,0], [y.min(), y.max()], [0,0], c='grey', ls='--', lw=0.5, alpha=0.3)
    
    return pca

print("✓ Función de graficación 3D definida")

## 8. Visualización 3D: BoW vs TF-IDF

In [ ]:
print("[8] Generando visualización 3D...\n")

# Crear figura con dos subplots 3D
fig = plt.figure(figsize=(18, 7))
fig.suptitle('Espacio Vectorial de Frankenstein: BoW vs TF-IDF (PCA 3D)', 
             fontsize=14, weight='bold', y=0.98)

# --- SUBPLOT 1: BAG OF WORDS ---
ax1 = fig.add_subplot(121, projection='3d')
pca_bow = graficar_palabras_3d(ax1, X_bow, vocab_bow, 
                               "Bag-of-Words (Conteos)", 
                               color_puntos='orange')

# --- SUBPLOT 2: TF-IDF ---
ax2 = fig.add_subplot(122, projection='3d')
pca_tfidf = graficar_palabras_3d(ax2, X_tfidf, vocab_tfidf, 
                                 "TF-IDF (Importancia Relativa)", 
                                 color_puntos='teal')

plt.tight_layout()
plt.savefig('frankenstein_vectorizacion_3d.png', dpi=300, bbox_inches='tight')
print("✓ Gráfico guardado como 'frankenstein_vectorizacion_3d.png'")
plt.show()

# Mostrar varianza explicada
print("\nVarianza Explicada por Componentes (PCA):")
print(f"\nBag-of-Words:")
for i, var in enumerate(pca_bow.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.1%}")
print(f"  Total: {pca_bow.explained_variance_ratio_.sum():.1%}")

print(f"\nTF-IDF:")
for i, var in enumerate(pca_tfidf.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.1%}")
print(f"  Total: {pca_tfidf.explained_variance_ratio_.sum():.1%}")

## 9. Análisis de Similitud entre Oraciones

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("[9] Análisis de Similitud de Coseno\n")

# Calcular similitud entre primeras 10 oraciones
n_oraciones = 10
similitud_bow = cosine_similarity(X_bow[:n_oraciones])
similitud_tfidf = cosine_similarity(X_tfidf[:n_oraciones])

# Visualizar como heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BoW
im1 = axes[0].imshow(similitud_bow, cmap='YlOrRd', aspect='auto')
axes[0].set_title('Similitud BoW (Primeras 10 oraciones)', weight='bold')
axes[0].set_xlabel('Oración')
axes[0].set_ylabel('Oración')
plt.colorbar(im1, ax=axes[0])

# TF-IDF
im2 = axes[1].imshow(similitud_tfidf, cmap='Greens', aspect='auto')
axes[1].set_title('Similitud TF-IDF (Primeras 10 oraciones)', weight='bold')
axes[1].set_xlabel('Oración')
axes[1].set_ylabel('Oración')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('frankenstein_similitud.png', dpi=300, bbox_inches='tight')
print("✓ Gráfico de similitud guardado como 'frankenstein_similitud.png'")
plt.show()

# Mostrar valores
print("\nMatriz de Similitud BoW (primeras 10 oraciones):")
print(pd.DataFrame(similitud_bow, columns=range(1,11), index=range(1,11)).round(3))

print("\nMatriz de Similitud TF-IDF (primeras 10 oraciones):")
print(pd.DataFrame(similitud_tfidf, columns=range(1,11), index=range(1,11)).round(3))

## 10. Resumen y Conclusiones

In [ ]:
print("="*70)
print("RESUMEN: VECTORIZACIÓN DE FRANKENSTEIN")
print("="*70)

resumen = f"""
📊 DATOS PROCESADOS:
  • Texto original: {len(texto_frankenstein):,} caracteres
  • Tokens totales: {len(doc):,}
  • Oraciones procesadas: {len(corpus_lematizado):,}
  • Vocabulario: 100 palabras más frecuentes (limitado)

📈 VECTORIZACIÓN:

  A. BAG-OF-WORDS (BoW):
     - Matriz: {X_bow.shape[0]} oraciones × {X_bow.shape[1]} palabras
     - Tipo: Conteos simples (¿cuántas veces aparece cada palabra?)
     - Ventaja: Simple, efectivo para clasificación
     - Desventaja: Pierde contexto y orden
     - Top palabra: '{vocab_bow[np.argsort(np.array(X_bow.sum(axis=0)).ravel())[-1]]}'

  B. TF-IDF:
     - Matriz: {X_tfidf.shape[0]} oraciones × {X_tfidf.shape[1]} palabras
     - Tipo: Ponderación por importancia (¿qué tan especial es cada palabra?)
     - Ventaja: Penaliza palabras comunes, destaca las distintivas
     - Desventaja: Más complejo, requiere todo el corpus
     - Top palabra: '{vocab_tfidf[np.argsort(np.array(X_tfidf.sum(axis=0)).ravel())[-1]]}'

🎨 VISUALIZACIÓN:
  • Espacio 3D con PCA (reducción de dimensionalidad)
  • Cada punto = una palabra
  • Proximidad = similaridad semántica en contexto
  • Archivos guardados:
    - frankenstein_vectorizacion_3d.png
    - frankenstein_similitud.png

🔍 INSIGHTS:
  • BoW destaca palabras frecuentes ("said", "upon", "could")
  • TF-IDF destaca palabras distintivas del Frankenstein
  • La similitud de coseno mide cercanía semántica entre oraciones
  • Valores altos = oraciones sobre temas similares

✅ FLUJO COMPLETO:
  Texto Crudo → Tokenización → Lematización → Vectorización → Visualización
"""

print(resumen)

## 11. Exportar Resultados

In [ ]:
# Guardar matrices en formato comprimido
import scipy.sparse

print("[10] Exportando resultados...\n")

# Guardar matrices sparse
scipy.sparse.save_npz('X_bow.npz', X_bow)
scipy.sparse.save_npz('X_tfidf.npz', X_tfidf)
print("✓ Matrices guardadas (X_bow.npz, X_tfidf.npz)")

# Guardar vocabularios
pd.DataFrame({
    'palabra': vocab_bow,
    'frecuencia_bow': np.array(X_bow.sum(axis=0)).ravel(),
    'frecuencia_tfidf': np.array(X_tfidf.sum(axis=0)).ravel()
}).to_csv('vocabulario_frankenstein.csv', index=False)
print("✓ Vocabulario guardado (vocabulario_frankenstein.csv)")

# Guardar corpus lematizado
with open('corpus_lematizado.txt', 'w', encoding='utf-8') as f:
    for i, oracion in enumerate(corpus_lematizado, 1):
        f.write(f"[{i}] {oracion}\n")
print("✓ Corpus lematizado guardado (corpus_lematizado.txt)")

print("\n✅ ¡Procesamiento completado exitosamente!")